# Audit podkladů a původního Overu
## tl;dr
Lokální rekonstrukce má neúplné a v závěru zastaralé xG podklady. Původní a přísnější Over sdílejí gólový model; jde o odlišné výběrové politiky, ne dva nezávislé modely. Žádná aktivace ani nový trénink.
## Context & Methods
Auditní companion, nikoli nový experiment. Pouze standardní Python; lokální zdroje a jejich SHA-256 vypíše výpočet. Ledger je k 18. 10. 2026, signály k 13. 10., rekonstrukce končí 19. 9. 2026.
### Key Assumptions
FT základní doba, jedna sázka = 1 jednotka; oddělené strategie/policy/model/context. Blokový interval ROI používá UTC herní dny (5 000 replik), u malého počtu dní je orientační. Výsledek není potvrzení ziskovosti ani nový holdout. Čas vydání statistik a původní quote timestamp nelze doložit.
## Data
Spouštět z kořene projektu nebo této složky; zdrojový skript obsahuje všechny kontroly a přesné cesty. Tabulkové výstupy zde slouží ke kontrole výpočtu, nikoli jako grafický report.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'research/ai/audit_data_and_over.py').exists())
sys.path.insert(0, str(root / 'research/ai'))
from audit_data_and_over import audit
report = audit()
print(report['cutoffs'])
print('Dataset:', report['profile']['n'], 'zápasů')
print('Vstupní hashe:', {k:v for k,v in report['sourceHashes'].items() if k.startswith('.runtime')})

## Results
### Integrita, společný jmenovatel a dostupnost

In [ ]:
assert all(v == 0 for v in report['invariants'].values())
assert not report['joinErrors'] and not report['resultMismatches']
assert not report['inventoryDuplicates'] and not report['inventoryConflicts']
print('Kontroly:', report['invariants'])
p = report['profile']
print('Obě xG:', p['bothXg'], '/', p['n'], '=', round(100*p['bothXg']/p['n'], 1), '%')
for part, values in report['xgAge'].items():
    print(part, values)

In [ ]:
print('Oddělené kohorty, simulace vkladu 1 jednotka:')
for key, c in report['cohorts'].items():
    assert c['pairedModel']['n'] == c['pairedMarket']['n']
    print(key, 'n=', c['n'], 'výhry=', c['hits'], 'ROI %=', round(100*c['roi'],2), '95% blokový interval=', c['roiDayBlock95'])
paired = report['pairedPolicies']
print('Společné fixture:', paired['n'], 'stejný čas:', paired['sameQualificationTime'], 'max rozdíl pravděpodobnosti:', paired['maxProbabilityDifference'])
for part, values in report['byPartition'].items():
    m = values['over']
    print(part, 'n=', m['n'], 'odhad Over %=', round(100*m['meanP'],2), 'skutečnost %=', round(100*m['actual'],2))

## Takeaways
Neplést dostupnost hodnoty s její čerstvostí. Další experiment musí měřit stáří vstupů a oddělit model bez pokročilých dat od kompletního profilu. Nejprve opravit datový kontrakt; nepřelaďovat pravidla na právě prohlédnutém terminalu. Profit rozdílných kohort není důkaz příčinného zlepšení modelu. Žádná data ani produkční historie se tímto notebookem nemění.